# Hofstra Data Science Week 3 - Exploratory Data Analysis (9/17/2026)

EDA means you look at the data before you model it. You check what is in it, what is missing, what is broken, and what patterns are worth chasing. Most modeling failures are really EDA failures that happened silently.

Today's data is a real Bloomberg export of about 1,600 US stocks, split across two files:

* `20260917_BLG_Port.xlsx`: valuation ratios, sector, market cap, returns, and factor exposures from Bloomberg's legacy US equity risk model
* `20260917_BLG_Port_2.xlsx`: buybacks, shares outstanding, capex estimates, quality scores, default risk, and factor exposures from **MAC3**, Bloomberg's current multi-asset risk model (17 style factors: momentum, value, growth, profitability, volatility, beta, quality, size, liquidity and more)

**Upload both files to this Colab session first** (folder icon on the left, then upload). These are licensed Bloomberg PORT exports and are not included in this repository; the README describes the columns so you can follow along with your own cross-section of stocks.

The question of the day: **what did the best 1-year returns look like, and which features line up with them?**

Plan: load and clean, **merge the two files**, describe the target, slice by sector, correlations, then a decile spread table.

## Load the data

Let's read the first file the naive way. **What do you notice?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df1 = pd.read_excel("20260917_BLG_Port.xlsx")
df1.head()

The real headers are not in the first row. Bloomberg puts two metadata rows on top (currency, index, adjustment flags). So we skip them. **How do we tell read_excel to skip rows?**

In [ ]:
df1 = pd.read_excel("20260917_BLG_Port.xlsx", skiprows=2)
df2 = pd.read_excel("20260917_BLG_Port_2.xlsx", skiprows=2)
df1.shape, df2.shape

In [ ]:
df1.head(3)

## Clean the junk rows

Bloomberg said 1,600 securities. We have 1,603 rows. **Where are the extra three?**

Last week's trick: rows that are not real stocks have no Short Name. Filter with `isna` / `notna`, never with `== None`.

In [ ]:
df1[df1["Short Name"].isna()][["Ticker"]]

One grouping row, one blank row, and the full Bloomberg legal disclaimer sitting in the last row. Real exports have footers. If you skip this check, the disclaimer becomes a data point.

In [ ]:
df1 = df1[df1["Short Name"].notna()]
df2 = df2[df2["Short Name"].notna()]
df1.shape, df2.shape

## Merge the two files

This is the first real step today. Two tables, one universe of stocks; we want one wide table.

Before any merge, pick the join key and **prove it is unique on both sides**. Candidates here: `Ticker`, which both files carry, or `CUSIP`, the official security identifier, which looks like the safer key. **Is it?**

In [ ]:
print("df1 duplicate Ticker:", df1["Ticker"].duplicated().sum())
print("df2 duplicate Ticker:", df2["Ticker"].duplicated().sum())
print("df1 duplicate CUSIP: ", df1["CUSIP"].duplicated().sum(), "| missing CUSIP:", df1["CUSIP"].isna().sum())
print("CUSIP in df2?", "CUSIP" in df2.columns)

The official identifier fails on both counts: 143 duplicate values in file 1 (144 CUSIPs are simply missing, and every missing value collides with every other missing value), and file 2 does not carry the column at all. The humble ticker is unique on both sides. The lesson: never assume a key is clean, check it.

Next, a dry run. An outer merge with `indicator=True` tells us who matches, who is only in the left file, and who is only in the right file, before we commit.

In [ ]:
probe = df1.merge(df2, on="Ticker", how="outer", indicator=True, suffixes=("_f1", "_f2"))
probe["_merge"].value_counts()

Every row matches. Now the real merge. `validate="one_to_one"` makes pandas throw an error if either side has duplicate keys, so silent row explosions cannot happen.

Last week's reflection question: **what do you check immediately after every merge?**

1. Row counts before and after
2. Duplicate keys (the `validate` argument does this for us)
3. The indicator dry run above
4. A column that exists in both files should agree between the two sides

In [ ]:
df = df1.merge(df2, on="Ticker", how="inner", suffixes=("_f1", "_f2"), validate="one_to_one")
print("rows before:", len(df1), "and", len(df2), "| rows after:", len(df))

In [ ]:
# check 4: both files carry yesterday's traded value; the two copies must agree
a = df["Equity Traded Val:D-1_f1"]
b = df["Equity Traded Val:D-1_f2"]
print("max abs difference:", (a - b).abs().max())

## Housekeeping: dtypes and column names

`describe` and correlations only work on numeric columns, so check what pandas thinks it has. Then rename the Bloomberg column names to short snake_case names, same idea as last week's rename.

The factor exposures come from two different Bloomberg models. File 1 has the legacy US equity model (renamed with a `us_` prefix); file 2 has MAC3, the current model (renamed `factor_`). The analysis below uses MAC3; the legacy columns stay for one comparison.

In [ ]:
df.dtypes.value_counts()

In [ ]:
# identifier and text columns stay text; force everything else numeric.
# Bloomberg sometimes ships numbers as text, and to_numeric(errors="coerce")
# turns any leftover junk like "#N/A" into NaN instead of crashing.
text_cols = ["Ticker", "Short Name_f1", "Short Name_f2", "Ticker.1_f1", "Ticker.1_f2",
             "CUSIP", "GICS SubInd", "GICS Sector", "GICS Ind Name", "GICS SubInd Name"]
for c in df.columns:
    if c not in text_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.rename(columns={
    "Short Name_f1": "name",
    "GICS Sector": "sector",
    "Total Return:Y-1": "return_1y",
    "Total Return:M-3": "return_3m",
    "Market Cap": "market_cap",
    # MAC3 factor exposures (file 2): the ones we analyze
    "PORT MAC3 Mom Factor Exp:D-1": "factor_momentum",
    "PORT MAC3 Value Factor Exp:D-1": "factor_value",
    "PORT MAC3 Valtn Factor Exp:D-1": "factor_valuation",
    "PORT MAC3 Ern Yld Factor Exp:D-1": "factor_earnings_yield",
    "PORT MAC3 Gwth Factor Exp:D-1": "factor_growth",
    "PORT MAC3 Prof Factor Exp:D-1": "factor_profitability",
    "PORT MAC3 Qual Factor Exp:D-1": "factor_quality",
    "PORT MAC3 Vol Factor Exp:D-1": "factor_volatility",
    "PORT MAC3 Residl Vol Factor Exp:D-1": "factor_residual_vol",
    "PORT MAC3 Beta Factor Exp:D-1": "factor_beta",
    "PORT MAC3 Lvrg Factor Exp:D-1": "factor_leverage",
    "PORT MAC3 Dvd Yld Factor Exp:D-1": "factor_dividend_yield",
    "PORT MAC3 Var Factor Exp:D-1": "factor_variability",
    "PORT MAC3 Size Factor Exp:D-1": "factor_size",
    "PORT MAC3 Mid Cap Factor Exp:D-1": "factor_mid_cap",
    "PORT MAC3 Lqdty Factor Exp:D-1": "factor_liquidity",
    "PORT MAC3 LT Rev Factor Exp:D-1": "factor_lt_reversal",
    # legacy US model exposures (file 1): kept for comparison only
    "PORT US Mom Fact Exp:D-1": "us_momentum",
    "PORT US Val Fact Exp:D-1": "us_value",
    "PORT US Gr Fact Exp:D-1": "us_growth",
    "PORT US Prof Fact Exp:D-1": "us_profitability",
    "PORT US Vol Fact Exp:D-1": "us_volatility",
    "PORT US Lev Fact Exp:D-1": "us_leverage",
    "PORT US Dvd Yld Fact Exp:D-1": "us_dividend_yield",
    "PORT US Earn Var Fact Exp:D-1": "us_variability",
    "PORT US Sz Fact Exp:D-1": "us_size",
    "FCF Yld": "fcf_yield",
    "P/E": "pe",
    "BEst P/E BF12M": "pe_forward",
    "P/B": "pb",
    "ROE LF": "roe",
    "Beta:Y-1": "beta",
    "Val Sh Repurch fr Repurch Prog:Y": "buyback_value",
    "Shares Out:Y": "shares_out",
    "Shares Out:Y-1": "shares_out_prior",
    "Bloomberg Issuer Likelihood of Default (Adjusted)": "default_prob",
    "BI STOX Multi-Factor Score (BLOOMBERG L.P.)": "stox_multifactor",
    "BI STOX Quality (BLOOMBERG L.P.)": "stox_quality",
    "BI STOX Low Volatility (BLOOMBERG L.P.)": "stox_low_vol",
    "BI STOX Momentum (BLOOMBERG L.P.)": "stox_momentum",
    "Number of Employees:Y": "employees",
    "Number of Employees:Y-5": "employees_5y_ago",
    "Insider Sells Volume 1 Year:Y-1": "insider_sells",
})
df[["Ticker", "name", "sector", "return_1y", "factor_momentum", "pb"]].head()

## Same name, different model

Both files carry a column called momentum, and value, and growth. Same word, two different models. **Would you expect them to agree?** Check before you assume: rank-correlate each legacy factor with its MAC3 namesake, and put each one's correlation with the 1-year return next to it.

In [ ]:
pairs = [("us_momentum", "factor_momentum"), ("us_value", "factor_value"), ("us_growth", "factor_growth"),
         ("us_profitability", "factor_profitability"), ("us_volatility", "factor_volatility"),
         ("us_leverage", "factor_leverage"), ("us_dividend_yield", "factor_dividend_yield"),
         ("us_variability", "factor_variability"), ("us_size", "factor_size")]
agree = pd.DataFrame({
    "legacy": [a for a, b in pairs],
    "mac3": [b for a, b in pairs],
    "legacy_vs_mac3": [df[a].corr(df[b], method="spearman") for a, b in pairs],
    "legacy_vs_return_1y": [df[a].corr(df["return_1y"], method="spearman") for a, b in pairs],
    "mac3_vs_return_1y": [df[b].corr(df["return_1y"], method="spearman") for a, b in pairs],
})
agree.round(2)

Seven of the nine agree at 0.6 to 0.9: value, leverage, dividend yield, variability, size, profitability, growth. Volatility agrees only at 0.41. Momentum agrees at **0.04**: the two columns called momentum are unrelated.

Now read the last two columns. MAC3 momentum tracks the 1-year return at 0.92; the legacy column tracks it at 0.13. Whatever the legacy model measured under that name, it was a different thing. A column name is a label, not a definition. Before you use a feature, find out what it is made of. The leakage section below is the payoff.

## Describe the target: 1-year total return

`describe` prints eight numbers. **Which row do you read first, and what decision does it drive?** (Reflection question 3 from last week.)

In [ ]:
df["return_1y"].describe()

Read it like a quant:

* `count` 1,558 of 1,600: forty-two stocks have no 1-year return. **Why would a stock have no 1-year return?** (Think IPOs.)
* mean +27% but median +12%: the average is dragged up by a fat right tail
* min is -99.8%: you can only lose 100%
* max is +1,560%: you can make 16x. That asymmetry is the whole shape of equity returns.

A histogram makes the skew visible.

In [ ]:
df["return_1y"].hist(bins=60)

In [ ]:
# the tail squashes everything; clip the x-range to see the body of the distribution
df["return_1y"].clip(-100, 200).hist(bins=60)

## Winners and losers

Sort and look. EDA is allowed to be this simple.

In [ ]:
cols = ["Ticker", "name", "sector", "return_1y"]
df.sort_values("return_1y", ascending=False)[cols].head(15)

In [ ]:
df.sort_values("return_1y")[cols].head(15)

**What do you notice about the winners?** Memory chips (Sandisk, Micron, AXT) and clinical-stage biotech; almost nothing boring. The losers are mostly tiny speculative names that fell 70 to 99%, with a few household names (Lucid, NuScale, Opendoor) among them.

## Slice by sector

`value_counts` for the census, then `groupby` for returns. We ask for mean **and** median. **Why both?**

In [ ]:
df["sector"].value_counts()

In [ ]:
df.groupby("sector")["return_1y"].agg(["count", "mean", "median"]).sort_values("median", ascending=False)

Health Care: mean +71% but median +32%. Half of health care did fine; a handful of biotechs going up 500 to 900% did the rest. The mean answers "what did a dollar spread across the sector earn", the median answers "what did the typical stock do". Different questions, different numbers.

A boxplot shows the spread inside each sector, and the outliers.

In [ ]:
df.boxplot(column="return_1y", by="sector", rot=90, figsize=(10, 5))

## Best performing sector, sub-industry, and segment

Sector is a coarse cut (11 buckets). GICS also gives sub-industries, which is where the story usually lives. We rank by **median** with a minimum count, so one moonshot cannot crown a bucket of three stocks.

In [ ]:
sub = (df.groupby("GICS SubInd Name")["return_1y"]
         .agg(["count", "median", "mean"])
         .query("count >= 8")
         .sort_values("median", ascending=False))
sub.head(12).round(1)

In [ ]:
sub.tail(8).round(1)

## The Morningstar style box

Morningstar sorts every stock into a 3 by 3 grid: size (large, mid, small) against style (value, blend, growth). We can build the same box from this data:

* size from market cap: large above $10B, mid $2B to $10B, small below $2B (Morningstar's real rule uses cumulative market cap percentiles, same idea)
* style from Bloomberg's factor exposures: growth minus value, cut into thirds

Then fill each of the nine cells with the median 1-year return. **Which box won the year?**

In [ ]:
df["size_box"] = pd.cut(df["market_cap"], bins=[0, 2e9, 10e9, np.inf], labels=["Small", "Mid", "Large"])
style_score = df["factor_growth"] - df["factor_value"]
df["style_box"] = pd.qcut(style_score, 3, labels=["Value", "Blend", "Growth"])

box = df.pivot_table(index="size_box", columns="style_box", values="return_1y", aggfunc="median", observed=True)
box = box.loc[["Large", "Mid", "Small"]]
counts = df.pivot_table(index="size_box", columns="style_box", values="return_1y", aggfunc="count", observed=True).loc[["Large", "Mid", "Small"]]
print("stocks per box:")
print(counts)

plt.figure(figsize=(6, 4.5))
sns.heatmap(box, annot=True, fmt=".0f", cmap="RdYlGn", center=0, cbar_kws={"label": "median 1-year return, %"})
plt.title("Style box: median 1-year return")
plt.ylabel("Size")
plt.xlabel("Style")
plt.show()
box.round(1)

Read it across and down. Across a row, style: does growth beat value at every size? Down a column, size: does large beat small at every style? A box where both directions agree is a real pattern; a single hot cell with a small count is noise. **Which cell would you have wanted to own a year ago, and would you have known to?**

Three things to notice. The whole Small row sits on 15 to 41 stocks (this universe's median market cap is about $10B, so under $2B is rare here): small counts, treat it as noise. In the Mid row, Growth at +22% beats Value at +9%; in the Large row the three cells sit within two points of each other, 12 to 14. So growth won in mid caps and nothing won in large caps. The decile table later in this notebook says value was one of the worst factors of the year and the growth factor also lost; the style box says the typical growth stock did fine. Both are true. The decile table uses means, which the biotech and chip moonshots pull up; the style box uses medians, which describe the typical stock. Ask which question you are answering before you pick the statistic.

## Missingness

Before trusting any feature, check how much of it exists.

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

The energy-intensity columns are missing for two thirds of the universe: ESG data is thin outside the largest companies. Buyback data exists for roughly half the universe, and that is real: many companies simply have no buyback program. Missing is information too.

## Which features line up with returns? Round 1: correlation

Correlate every numeric column with `return_1y` and sort. Keep expectations low: stock returns are mostly noise, so in this world an absolute correlation of 0.2 is a big deal. One number in the output will break that rule by a mile. Hold that thought.

In [ ]:
num = df.select_dtypes("number")
corr = num.corrwith(df["return_1y"]).drop(["return_1y"], errors="ignore")
corr = corr[~corr.index.str.startswith("us_")]     # legacy-model columns are out of scope here
corr = corr.sort_values(ascending=False)
print("Most positive:")
print(corr.head(10).to_string())
print()
print("Most negative:")
print(corr.tail(10).to_string())

## Correlation heat map

A sorted list shows each feature against the target. A heat map shows the features against each other, which is how you find the ones that are secretly the same thing. We use Spearman (rank) correlation here rather than Pearson, because a +1,599% return would otherwise dominate every Pearson number. **Which pairs are near duplicates?**

In [ ]:
heat_cols = ["return_1y", "return_3m", "factor_momentum", "factor_value", "factor_growth",
             "factor_profitability", "factor_quality", "factor_volatility", "factor_beta", "factor_leverage",
             "factor_dividend_yield", "factor_size", "pb", "pe", "roe", "beta", "fcf_yield", "market_cap"]
corr_matrix = df[heat_cols].corr(method="spearman")

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Spearman correlation, 1,600 US stocks, MAC3 factors")
plt.show()

Reading a heat map: look for the dark cells away from the diagonal first. Size and market cap at 0.95 (the same thing, measured twice); MAC3 beta and Bloomberg's plain beta at 0.93; volatility and beta at 0.85; profitability and ROE at 0.81; the value factor against P/E at -0.80 and P/B at -0.65 (value scores cheapness, so it runs opposite to the price multiples); and momentum against the 1-year return at 0.92. When two features are near duplicates, keep one; the model will not learn twice from the same number, and the second one only makes the results harder to explain.

## Leakage alarm

Momentum sits at the top of every list: correlation 0.92 with the 1-year return. **Before you get excited, ask: how is the momentum factor computed?**

MAC3 builds it from the stock's own trailing returns over roughly the past year. We are correlating last year's return with a function of last year's return. Of course they agree. This is called leakage: the feature secretly contains the answer. The legacy model's momentum column correlated only 0.13 with the same return, which is why the two momentum columns disagreed with each other at 0.04: they were never measuring the same thing. When a feature looks too good, the first question is what the feature is made of, the same way the 150-year-old Social Security recipients came from knowing how COBOL stores a missing date.

In [ ]:
# regplot = scatter plus a fitted straight line; returns clipped so the tail does not squash the picture
sns.regplot(x="factor_momentum", y=df["return_1y"].clip(-100, 300), data=df,
            scatter_kws={"alpha": 0.25, "s": 12}, line_kws={"color": "red"})
plt.ylabel("return_1y (clipped to -100..300)")
plt.show()

## Scatter plots with a regression line

The heat map gives one number per pair. A scatter plot shows the shape behind the number, and the fitted line shows the slope. Four features against the 1-year return, returns clipped to -100..300 so the biotech moonshots do not flatten everything else.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
pairs = [("pb", "Price to book"), ("fcf_yield", "Free cash flow yield"),
         ("beta", "Beta"), ("log_market_cap", "log10 market cap")]
df["log_market_cap"] = np.log10(df["market_cap"])
y = df["return_1y"].clip(-100, 300)
for ax, (col, label) in zip(axes.ravel(), pairs):
    x = df[col].clip(df[col].quantile(0.01), df[col].quantile(0.99))   # trim the 1% tails on x too
    sns.regplot(x=x, y=y, ax=ax, scatter_kws={"alpha": 0.25, "s": 10}, line_kws={"color": "red"})
    ax.set_xlabel(label)
    ax.set_ylabel("1-year return (clipped)")
plt.tight_layout()
plt.show()

**Where is the line lying to you?** A straight line through a cloud this wide is a weak summary; the slope is real but the scatter around it is enormous, which is exactly what a correlation of 0.2 looks like. Two habits: clip or rank before you fit (a single 16x stock can tilt the line by itself), and always look at the cloud, never just the slope. The next section makes the same point with a feature that looks stronger than it is.

## Make your own features

Feature engineering is EDA too. Two ideas from the buyback file:

* `buyback_yield`: dollars spent on buybacks divided by market cap
* `share_count_change`: shares outstanding this year vs last year (buybacks shrink it, issuance grows it)

In [ ]:
df["buyback_yield"] = df["buyback_value"] / df["market_cap"]
df["share_count_change"] = df["shares_out"] / df["shares_out_prior"] - 1
df[["buyback_yield", "share_count_change"]].describe()

Look at the max of `share_count_change`: over one million. That is a data artifact (a prior share count near zero), which would wreck any average. **How do we explore a feature without letting one broken value dominate?** Use ranks and buckets instead of raw values.

## Round 2: decile spreads, the punchline machine

For each feature: rank the stocks, cut them into 10 equal buckets with `qcut`, take the mean return of bucket 10 minus bucket 1. That number says: last year, how differently did the top tenth and bottom tenth of this feature perform?

In [ ]:
def decile_spread(col):
    # rank first so qcut always finds 10 clean buckets even with ties or crazy outliers
    dec = pd.qcut(df[col].rank(method="first"), 10, labels=False) + 1
    m = df.groupby(dec)["return_1y"].mean()
    return m.iloc[-1] - m.iloc[0]

# one feature end to end first: momentum, bucket by bucket
dec = pd.qcut(df["factor_momentum"].rank(method="first"), 10, labels=False) + 1
df.groupby(dec)["return_1y"].mean()

In [ ]:
candidates = ["pb", "pe", "roe", "beta", "fcf_yield", "market_cap",
              "factor_momentum", "factor_value", "factor_valuation", "factor_earnings_yield",
              "factor_growth", "factor_profitability", "factor_quality",
              "factor_volatility", "factor_residual_vol", "factor_beta", "factor_leverage",
              "factor_dividend_yield", "factor_variability", "factor_size", "factor_mid_cap",
              "factor_liquidity", "factor_lt_reversal",
              "stox_multifactor", "stox_quality", "stox_low_vol", "stox_momentum", "default_prob",
              "buyback_yield", "share_count_change", "insider_sells",
              "us_momentum"]   # the legacy momentum stays in as a control; see the leakage section

rows = []
for c in candidates:
    if df[c].notna().sum() < 300:      # skip features that barely exist
        continue
    rows.append({"feature": c, "spread": decile_spread(c), "n": df[c].notna().sum()})

pd.DataFrame(rows).sort_values("spread", ascending=False)

## Read the table

**What kind of year was it?**

* Momentum leads at 238 points and the STOX momentum score at 211. Ignore both: that is the leakage from the previous section, and the legacy momentum column at 50 points shows what a momentum-flavoured feature without the leak looks like.
* Real top of the table: volatility, residual volatility, beta, P/B. Expensive, fast-moving, speculative stocks won by 50 to 70 points.
* Bottom of the table: valuation, low volatility, value, leverage, earnings yield. Cheap, calm, cash-generating stocks lost by 50 to 80 points. The growth factor, which scores companies on delivered earnings and sales growth, lost 22 points too: the winners were expensive, not fast-growing by the numbers.
* Buyback yield is negative and share issuance is positive: companies printing new shares beat companies buying their own shares back. The market paid for dilution.

Before we believe any of it: **is a handful of strange stocks driving this table?**

## Outliers

Three kinds of outliers show up in real data:

1. **Data errors**, like the one-million-x share count change. Fix or drop.
2. **Real extreme events**, like a +1,599% year. Keep them, but do not let them drive your averages.
3. **Unusual combinations**: a stock that looks normal on every single column but strange when you look at all the columns together. You cannot see these one histogram at a time.

Start with the textbook one-column rule: anything beyond 1.5 IQR outside the quartiles. **How many stocks does it flag?**

In [ ]:
q1, q3 = df["return_1y"].quantile([0.25, 0.75])
iqr = q3 - q1
iqr_flag = (df["return_1y"] < q1 - 1.5 * iqr) | (df["return_1y"] > q3 + 1.5 * iqr)
iqr_flag.sum()

108 stocks, 7% of the universe. The IQR rule assumes a bell-shaped column. Stock returns have fat tails, so the rule calls the entire tail an outlier. Useless here.

## Multivariate outliers with PyOD

[PyOD](https://pyod.readthedocs.io/) is the standard Python library for outlier detection. Do you need it? scikit-learn already ships Isolation Forest. PyOD adds two things worth one install cell: one interface to fifty-plus detectors, and ECOD, a detector with no parameters to tune that can explain each flag feature by feature.

We run two different detectors and keep the stocks that both flag. Agreement between methods is cheap insurance.

In [ ]:
!pip install pyod -q

Detectors cannot take NaN. We fill missing values with the column median, so a missing value looks average rather than extreme (dropping rows would throw away every IPO). Market cap goes in as log10 so a few trillion-dollar companies do not dominate the scale.

In [ ]:
import numpy as np
from pyod.models.ecod import ECOD
from pyod.models.iforest import IForest

df["log_market_cap"] = np.log10(df["market_cap"])
feat = ["return_1y", "return_3m", "pb", "pe", "roe", "beta", "fcf_yield", "log_market_cap",
        "share_count_change", "factor_momentum", "factor_value", "factor_growth",
        "factor_profitability", "factor_volatility", "factor_leverage",
        "factor_dividend_yield", "factor_variability", "factor_quality", "factor_size"]
X = df[feat].fillna(df[feat].median())

# contamination = the share of the data we are willing to call outliers (2% of 1,600 = 32 stocks)
ecod = ECOD(contamination=0.02).fit(X)
iforest = IForest(contamination=0.02, random_state=42).fit(X)

df["ecod_score"] = ecod.decision_scores_
df["ecod_flag"] = ecod.labels_.astype(bool)
df["iforest_flag"] = iforest.labels_.astype(bool)
df["outlier"] = df["ecod_flag"] & df["iforest_flag"]
print("ECOD flagged:", df["ecod_flag"].sum(), "| Isolation Forest flagged:", df["iforest_flag"].sum(),
      "| flagged by both:", df["outlier"].sum())

In [ ]:
cols = ["Ticker", "name", "sector", "return_1y", "pb", "share_count_change", "ecod_score"]
df[df["outlier"]].sort_values("ecod_score", ascending=False)[cols]

## Ask the model why

ECOD's score is a sum over the 19 features of how deep in the tail each value sits (minus the log of the empirical CDF, on whichever side is more extreme). The most any single feature can contribute is about 7.4, the single most extreme value out of 1,600. So every flagged stock's score splits into feature pieces, and we can read the model's own reasons.

In [ ]:
n = len(X)
O = pd.DataFrame(ecod.O[-n:], columns=feat, index=df.index)                            # per-stock, per-feature tail scores
low_side = pd.DataFrame(ecod.U_l[-n:] >= ecod.U_r[-n:], columns=feat, index=df.index)  # True where the low tail is the extreme side
pct = X.rank(pct=True)                                                                  # percentile of each value in the universe

print("max possible contribution from one feature:", round(-np.log(1 / n), 2))

rows = []
for i in df.index[df["outlier"]]:
    top = O.loc[i].sort_values(ascending=False).head(3)
    why = []
    for f, c in top.items():
        side = "low" if low_side.loc[i, f] else "high"
        why.append(f"{f} {side} (pct {pct.loc[i, f] * 100:.1f}, {c:.1f})")
    rows.append({"Ticker": df.loc[i, "Ticker"], "score": round(df.loc[i, "ecod_score"], 1),
                 "top3_share": round(top.sum() / O.loc[i].sum(), 2), "why": " | ".join(why)})

pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows).sort_values("score", ascending=False)

Two things in the model output that a human reading would miss:

* The top three features explain only a quarter to a third of any score. These stocks are not extreme on one column; they sit in the tail on eight or nine columns at once. That is what a multivariate outlier means.
* The most frequent drivers are 1-year return, ROE, market cap, 3-month return, quality, and FCF yield. A collapse pulls all of them into the tail together: the price falls, so market cap falls, yields flip sign, and both returns crater.

Now put a human reading next to the model's. **Which of the three kinds is each one?**

* The -90% collapses (Future Fintech, Antelope, ReTo, Jinxin, DataMeds, Synergy CHC, Tempest): real events, tiny companies that fell apart.
* Share count changes of 228x (Q/C Technologies) and 482x (SharpLink), 48x (ReTo), 12x (Beneficient): data artifacts from reverse splits or bad prior share counts. Notice the model did not need the share count to flag ReTo; its FCF yield, P/B and beta are all in the bottom 0.2% on their own.
* The moonshots (Sandisk +1,560%, Cypherpunk +669%, Hut 8 +156%, T1 Energy +134%): real extreme events.
* A cluster of crypto-linked names: SharpLink, Strive, Galaxy Digital, Hut 8, TeraWulf, Cypherpunk. Nothing is wrong with those rows. They move together and unlike everything else, so every detector sees them as a group living in its own corner.

An outlier is a stock that is unusual, not a stock that is wrong. Never delete rows because a model flagged them. Look first.

ECOD can show **why** any stock scores the way it does: one bar per feature, showing how far into the tail that stock's value sits. Let's ask about Nvidia, which is **not** flagged this time (ECOD ranks it 113th of 1,600): up 23%, P/B of 23, share count going down. It sits in the tail on size, profitability and valuation at the same time, and that is still not enough to make the list. Unusual is not the same as extreme.

In [ ]:
pos = list(df["Ticker"]).index("NVDA US Equity")
ecod.explain_outlier(pos, feature_names=feat)

## Robustness check: does the story survive without the outliers?

This is the whole point of finding them. Recompute the decile spreads with the flagged stocks removed and compare.

In [ ]:
def decile_spread_on(data, col):
    dec = pd.qcut(data[col].rank(method="first"), 10, labels=False) + 1
    m = data.groupby(dec)["return_1y"].mean()
    return m.iloc[-1] - m.iloc[0]

clean = df[~df["outlier"]]
rows = []
for c in candidates:
    if df[c].notna().sum() < 300:
        continue
    rows.append({"feature": c,
                 "spread_all": decile_spread_on(df, c),
                 "spread_no_outliers": decile_spread_on(clean, c)})
compare = pd.DataFrame(rows)
compare["change"] = compare["spread_no_outliers"] - compare["spread_all"]
compare.sort_values("spread_all", ascending=False).round(1)

Same story at both ends. Momentum shrinks by 17 points, beta and P/B by about 12, and market cap and size lose about 18 each (the flagged set is mostly tiny collapses plus Sandisk, so it sat at the ends of the size ranking). Leverage moves the most of the losers, 12 points less negative. The order of the table barely moves: the same five at the top, the same three at the bottom. That is what robust means: the conclusion does not depend on a few rows.

## Caveats before anyone trades on this

1. **Direction of time.** The features are measured today, the returns already happened. This describes what winners look like now; it does not prove the features predicted anything.
2. **Survivorship.** Today's universe only contains stocks that survived to today.
3. **One year is one draw.** In a different year this table can flip completely, and the low-volatility line usually points the other way.

EDA gives you the shape of the data and a list of suspects. It never gives you a verdict.

## Challenge problem

Write a function `spread_by_sector(col)` that computes the decile spread of a feature **within each sector** (hint: `groupby("sector")` plus the `decile_spread` idea, but rank within the sector). Then answer: does the P/B result hold in every sector, or is it really just one or two sectors driving it? Bring your answer next week; explaining it in class earns the extra credit.

Bonus: rerun the two detectors with `contamination=0.05` (80 stocks) and check whether the robustness table still tells the same story.